# Linear Regression: Airquality

All tests are independent editable blocks. For vectors, follow the printed coefficient order.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

PLOTS_DIR = ROOT / "plots"
PLOTS_DIR.mkdir(exist_ok=True)

def find_data_file(file_name):
    candidates = [ROOT / file_name, ROOT / "data" / file_name]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(
        f"Could not find {file_name}. Put it in the project root or data/ folder and include the extension."
    )

def save_plot(fig, file_name):
    path = PLOTS_DIR / file_name
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved plot: {path}")
    return path

from regression_toolbox import RegressionToolbox

use_default_dataset = True
custom_file_name = "my_regression_data.csv"

data_path = ROOT / "data" / "airquality.csv" if use_default_dataset else find_data_file(custom_file_name)
tool = RegressionToolbox(data_path)
df = tool.data.copy()

y_col = "Ozone"
simple_x_col = "Temp"
x_cols = ["Temp", "Wind", "Solar.R"]
needed = list(dict.fromkeys([y_col, simple_x_col] + x_cols))
missing = [c for c in needed if c not in df.columns]
if missing:
    raise ValueError(f"Set y_col/simple_x_col/x_cols to columns in your data. Missing: {missing}. Available: {list(df.columns)}")
clean_rows = df[needed].apply(pd.to_numeric, errors="coerce").dropna()
if len(clean_rows) <= len(x_cols) + 1:
    raise ValueError("Regression needs more complete numeric rows than coefficients.")

print("Dataset:", data_path)
print("Shape:", df.shape)
display(df.head())
display(df.isna().sum().to_frame("missing_values"))
display(tool.summarize_data(needed))

In [ ]:
# Editable block: simple linear regression fit
simple_x_col = "Temp"
y_col = "Ozone"
alpha = 0.05

simple = tool.fit_simple_linear_regression(simple_x_col, y_col, alpha)
print("Coefficient order:", simple["coefficient_names"])
display(tool.coefficient_table(simple))
display(tool.anova_regression_table(simple))
fig, ax = tool.plot_simple_regression(simple)
save_plot(fig, "regression_simple_fit.png")
fig, ax = tool.plot_observed_vs_fitted(simple)
save_plot(fig, "regression_simple_observed_vs_fitted.png")
fig, ax = tool.plot_residuals_vs_fitted(simple)
save_plot(fig, "regression_simple_residuals_vs_fitted.png")

In [ ]:
# Editable block: test intercept a
coefficient = "Intercept"
null_value = 0
alternative = "two-sided"
alpha = 0.05

test = tool.coefficient_t_test(simple, coefficient, null_value, alternative, alpha, label="Test intercept a")
display(pd.DataFrame([test]))
fig, ax = tool.plot_t_distribution_for_coefficient(test)
save_plot(fig, "regression_intercept_a_t_test.png")

In [ ]:
# Editable block: test slope beta
coefficient = simple_x_col
null_value = 0
alternative = "greater"
alpha = 0.05

test = tool.coefficient_t_test(simple, coefficient, null_value, alternative, alpha, label="Test slope beta")
display(pd.DataFrame([test]))
fig, ax = tool.plot_t_distribution_for_coefficient(test)
save_plot(fig, "regression_slope_beta_t_test.png")

In [ ]:
# Editable block: multiple linear regression fit
x_cols = ["Temp", "Wind", "Solar.R"]
y_col = "Ozone"
alpha = 0.05

model = tool.fit_multiple_linear_regression(x_cols, y_col, alpha)
print("Coefficient order:", model["coefficient_names"])
display(tool.coefficient_table(model))
display(tool.anova_regression_table(model))
fig, ax = tool.plot_observed_vs_fitted(model)
save_plot(fig, "regression_multiple_observed_vs_fitted.png")
fig, ax = tool.plot_residuals_vs_fitted(model)
save_plot(fig, "regression_multiple_residuals_vs_fitted.png")

In [ ]:
# Editable block: multiple regression coefficient test
coefficient = "Wind"
null_value = 0
alternative = "less"
alpha = 0.05

test = tool.coefficient_t_test(model, coefficient, null_value, alternative, alpha, label="Multiple regression coefficient test")
display(pd.DataFrame([test]))
fig, ax = tool.plot_t_distribution_for_coefficient(test)
save_plot(fig, "regression_multiple_coefficient_t_test.png")

In [ ]:
# Editable block: test part of beta using c'beta
print("Coefficient order:", model["coefficient_names"])
contrast_vector = [0, 1, 0, 0]
null_value = 0
alternative = "two-sided"
alpha = 0.05

test = tool.linear_combination_test(model, contrast_vector, null_value, alternative, alpha, label="Test c'beta")
display(pd.DataFrame([test]))
fig, ax = tool.plot_t_distribution_for_linear_combination(test)
save_plot(fig, "regression_linear_combination_t_test.png")

In [ ]:
# Editable block: confidence interval for c'beta
contrast_vector = [0, 1, 0, 0]
confidence = 0.95

ci = tool.linear_combination_ci(model, contrast_vector, confidence, label="Linear combination CI")
display(pd.DataFrame([ci]))

In [ ]:
# Editable block: general linear F-test C beta = d
print("Coefficient order:", model["coefficient_names"])
C = [[0, 0, 1, 0], [0, 0, 0, 1]]
d = [0, 0]
alpha = 0.05

test = tool.general_linear_f_test(model, C, d, alpha, label="General linear F-test for part of beta")
display(pd.DataFrame([test]))
fig, ax = tool.plot_f_distribution_for_model(test)
save_plot(fig, "regression_general_linear_f_test.png")

In [ ]:
# Editable block: overall regression F-test
alpha = 0.05

test = tool.overall_f_test(model, alpha, label="Overall regression F-test")
display(pd.DataFrame([test]))
fig, ax = tool.plot_f_distribution_for_model(test)
save_plot(fig, "regression_overall_f_test.png")

In [ ]:
# Editable block: mean response and prediction intervals
new_data = pd.DataFrame({"Temp": [70, 80, 90], "Wind": [8, 10, 12], "Solar.R": [150, 200, 250]})
confidence = 0.95

display(tool.predict(model, new_data, confidence, interval="mean").assign(interval="mean"))
display(tool.predict(model, new_data, confidence, interval="prediction").assign(interval="prediction"))

In [ ]:
# Editable block: Scheffe simultaneous mean response band
alpha = 0.05

band = tool.scheffe_mean_response_band(simple, alpha=alpha)
display(band.head())
fig, ax = tool.plot_scheffe_band(simple, band, title="Scheffe simultaneous mean response band")
save_plot(fig, "regression_scheffe_mean_response_band.png")

In [ ]:
# Editable block: Scheffe-style prediction-band variant
# This is labeled as a prediction-band variant because Scheffe's standard band is for the mean response.
alpha = 0.05

band = tool.scheffe_prediction_band(simple, alpha=alpha)
display(band.head())
fig, ax = tool.plot_scheffe_band(simple, band, title="Scheffe-style prediction-band variant")
save_plot(fig, "regression_scheffe_prediction_band_variant.png")